# Extract Opposition Managers

Read each file in `bbc-json/lineups`, identify Tranmere Rovers, and capture the opposing manager's BBC id and name.

In [1]:
from __future__ import annotations

import csv
import json
from pathlib import Path

LINEUPS_DIR = Path('bbc-json/lineups')
OUTPUT_CSV = Path('data-r/bbc_opposition_managers.csv')
TRANMERE_URN = 'urn:bbc:sportsdata:football:team:tranmere-rovers'
TRANMERE_NAME = 'Tranmere Rovers'

In [2]:
def is_tranmere(team: dict) -> bool:
    if not team:
        return False

    team_urn = team.get('urn')
    full_name = team.get('name', {}).get('fullName')
    return team_urn == TRANMERE_URN or full_name == TRANMERE_NAME


def extract_opponent_manager(file_path: Path) -> dict | None:
    with file_path.open(encoding='utf-8') as handle:
        match_data = json.load(handle)

    home_team = match_data.get('homeTeam', {})
    away_team = match_data.get('awayTeam', {})

    if is_tranmere(home_team):
        opponent = away_team
    elif is_tranmere(away_team):
        opponent = home_team
    else:
        return None

    manager = opponent.get('manager', {})
    return {
        'game_date': file_path.stem,
        'bbc_id': manager.get('id'),
        'manager_name': manager.get('name', {}).get('full'),
    }

In [3]:
rows = []

for file_path in sorted(LINEUPS_DIR.glob('*.json')):
    row = extract_opponent_manager(file_path)
    if row is not None:
        rows.append(row)

rows[:5], len(rows)

([{'game_date': '2020-01-01',
   'bbc_id': 'cjvjlzuwjsp1t9f2c1kpt77x1',
   'manager_name': 'Mark Robins'},
  {'game_date': '2020-01-04',
   'bbc_id': '5rnibm58n3dbos9rsz1lahbyt',
   'manager_name': 'Nigel Pearson'},
  {'game_date': '2020-01-07',
   'bbc_id': '2az1hqn1ylzk3zshjerwjtuxh',
   'manager_name': 'Steve Beaglehole'},
  {'game_date': '2020-01-11',
   'bbc_id': 'bw62bse00i2uhalntu7t1dp05',
   'manager_name': 'Sol Campbell'},
  {'game_date': '2020-01-18',
   'bbc_id': 'ziigr8o4c81c0bbjxugpozx1',
   'manager_name': 'Paul Lambert'}],
 332)

In [4]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

with OUTPUT_CSV.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=['game_date', 'bbc_id', 'manager_name'])
    writer.writeheader()
    writer.writerows(rows)

print(f'Wrote {len(rows)} rows to {OUTPUT_CSV}')
rows[:10]

Wrote 332 rows to data-r/bbc_opposition_managers.csv


[{'game_date': '2020-01-01',
  'bbc_id': 'cjvjlzuwjsp1t9f2c1kpt77x1',
  'manager_name': 'Mark Robins'},
 {'game_date': '2020-01-04',
  'bbc_id': '5rnibm58n3dbos9rsz1lahbyt',
  'manager_name': 'Nigel Pearson'},
 {'game_date': '2020-01-07',
  'bbc_id': '2az1hqn1ylzk3zshjerwjtuxh',
  'manager_name': 'Steve Beaglehole'},
 {'game_date': '2020-01-11',
  'bbc_id': 'bw62bse00i2uhalntu7t1dp05',
  'manager_name': 'Sol Campbell'},
 {'game_date': '2020-01-18',
  'bbc_id': 'ziigr8o4c81c0bbjxugpozx1',
  'manager_name': 'Paul Lambert'},
 {'game_date': '2020-01-23',
  'bbc_id': '5rnibm58n3dbos9rsz1lahbyt',
  'manager_name': 'Nigel Pearson'},
 {'game_date': '2020-01-26',
  'bbc_id': '1jt6jnwdiaicjbff0phe6w2xh',
  'manager_name': 'Ole Gunnar Solskjær'},
 {'game_date': '2020-01-29',
  'bbc_id': '1q7vpureb5sarl7lz06jshw45',
  'manager_name': 'Phil Parkinson'},
 {'game_date': '2020-02-01',
  'bbc_id': 'f1q9ez4eg94jjmf4oa593wwr9',
  'manager_name': 'Keith Hill'},
 {'game_date': '2020-02-04',
  'bbc_id': 'b9